In [ ]:
from pathlib import Path
import cl
from cl import RecordingView
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA, NMF

rng = np.random.default_rng(0)

In [ ]:
recording = RecordingView("/mnt/data/cl/2026-04-15_22-58-53.129+00-00_worm_training.h5")

In [ ]:
channel_positions = np.arange(64).reshape((8,8)).T

In [ ]:
s_c = recording.spikes.col('channel')
s_t = recording.spikes.col('timestamp') / recording.attributes['frames_per_second']  # units are s

In [ ]:
st_c = recording.stims.col('channel')
st_t = recording.stims.col('timestamp') / recording.attributes['frames_per_second']

In [ ]:

y, x = np.unravel_index(s_c, channel_positions.shape)
fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))
ax.scatter(x, y, s_t, s=1)
ax.set_zlim([0,60])

In [ ]:
%matplotlib inline
recording.plot_spikes_and_stims()

In [ ]:
r = recording.analyse_functional_connectivity(bin_size_sec=.1, correlation_threshold=.1)
r.plot()
# plt.matshow(r.adjacency_matrix)

In [ ]:
%matplotlib inline

step = 0.01
bin_edges = np.arange(-.5, .5+step, step)
jitter = 0

fig, axs = plt.subplots(nrows=4, ncols=4, constrained_layout=True, figsize=(10,10))

for outer_i in range(axs.shape[0]):
    for outer_j in range(axs.shape[1]):
        if outer_j > outer_i:
            continue
        c1 = np.argsort(a.sum(axis=0))[-1 - outer_i]
        c2 = np.argsort(a.sum(axis=0))[-1 - outer_j]

        spike_times1 = s_t[s_c == c1]
        spike_times2 = s_t[s_c == c2]

        if jitter:
            spike_times1 = spike_times1 + np.random.normal(0, jitter, spike_times1.shape)
            spike_times2 = spike_times2 + np.random.normal(0, jitter, spike_times2.shape)

        bins = np.zeros(len(bin_edges) + 1)

        left_bound = 0
        for i, t1 in enumerate(spike_times1):
            for j in range(left_bound, len(spike_times2)):
                if spike_times2[j] < t1 + bin_edges[0]:
                    left_bound = j
                elif spike_times2[j] > t1 + bin_edges[-1]:
                    break
                else:
                    same_spike = c1 == c2 and i == j
                    if not same_spike:
                        bins[np.searchsorted(bin_edges, t1 - spike_times2[j])] += 1


        axs[outer_i, outer_j].bar(bin_edges, bins[:-1], width=step)

        bins = bins[1:-1]
        from scipy.stats import multinomial
        log_p = multinomial.logpmf(bins, bins.sum(), np.ones(len(bins)) / len(bins))
        # axs[outer_i, outer_j].set_title(f'{log_p=:.1f}')
        axs[outer_i, outer_j].set_title(f'{c1} -> {c2}')



In [ ]:
%matplotlib inline

step = 0.02
bin_edges = np.arange(-.5, .5+step, step)
jitter = 0

fig, axs = plt.subplots(nrows=6, ncols=6, constrained_layout=True, figsize=(10,10))

for outer_i in range(axs.shape[0]):
    for outer_j in range(axs.shape[1]):
        spike_channel = np.argsort(a.sum(axis=0))[-1 - outer_i]
        stim_channel = np.unique(st_c)[outer_j]

        spike_times1 = s_t[s_c == spike_channel]
        stim_times = st_t[st_c == stim_channel]
        # stim_times = np.sort(rng.uniform(low=s_t.min(),high=s_t.max(),  size=(st_c == stim_channel).sum()))

        if jitter:
            spike_times1 = spike_times1 + np.random.normal(0, jitter, spike_times1.shape)
            stim_times = stim_times + np.random.normal(0, jitter, stim_times.shape)

        bins = np.zeros(len(bin_edges) + 1)

        left_bound = 0
        for i, t1 in enumerate(spike_times1):
            for j in range(left_bound, len(stim_times)):
                if stim_times[j] < t1 + bin_edges[0]:
                    left_bound = j
                elif stim_times[j] > t1 + bin_edges[-1]:
                    break
                else:

                    bins[np.searchsorted(bin_edges, t1 - stim_times[j])] += 1


        axs[outer_i, outer_j].bar(bin_edges, bins[:-1], width=step)

        bins = bins[1:-1]
        from scipy.stats import multinomial
        log_p = multinomial.logpmf(bins, bins.sum(), np.ones(len(bins)) / len(bins))
        # axs[outer_i, outer_j].set_title(f'{log_p=:.1f}')
        axs[outer_i, outer_j].set_title(f'{stim_channel} -> {spike_channel}')



In [ ]:
a_bins = np.arange(s_t[0], s_t[-1], 50 / 1000)
bins = a_bins
a = np.array([np.histogram(s_t[s_c == channel], bins=bins)[0] for channel in range(64)]).T
a_nz = a[:, a.sum(axis=0) > 30 * 1]
a_nz = a_nz[:,np.argsort(a_nz.sum(axis=0))[::-1]]

In [ ]:
cov = np.cov(a_nz.T)
off_diagonals = cov - np.diag(np.diag(cov))
plt.matshow(np.abs(cov))
# plt.colorbar()

In [ ]:
unique_rows, row_counts = np.unique(a, axis=0, return_counts=True)
idx = np.argsort(row_counts)[::-1]
unique_rows, row_counts = unique_rows[idx], row_counts[idx]
idx = row_counts > 1
unique_rows, row_counts = unique_rows[idx], row_counts[idx]
print(len(row_counts))

In [ ]:
transitions = np.zeros((len(unique_rows), len(unique_rows)))


old_idx = np.argmin(np.linalg.norm(unique_rows - a[0], axis=1))
for i in range(1,len(a)):
    new_idx = np.argmin(np.linalg.norm(unique_rows - a[i], axis=1))
    transitions[old_idx, new_idx] += 1
    old_idx = new_idx

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(5,4))
c = ax.matshow(np.log10(transitions[:20,:20]))
cbar = plt.colorbar(c)
ax.set_ylabel('from')
ax.set_xlabel('to')

cbar.set_label('log10(count)', rotation=270, labelpad=20)


In [ ]:
import networkx as nx
G = nx.from_numpy_array(transitions)

nx.draw_spring(G)

In [ ]:
plt.matshow(unique_rows[:10])

In [ ]:
k = np.exp(np.linspace(0, -5, 300))
k = k / np.sum(k)
# k = [1]
smoothed_a = np.apply_along_axis(lambda x: np.convolve(x, k, mode='same'), 0, a)
plt.matshow(smoothed_a[:25])

In [ ]:
%matplotlib inline
plt.plot(smoothed_a)
plt.plot(a)
plt.xlim([0, 5000/40])

In [ ]:
pca = PCA(n_components=4)
latents = pca.fit_transform(smoothed_a)

In [ ]:
%matplotlib inline
fib, ax = plt.subplots()
ax.plot(latents[:, 0], latents[:, 1])

In [ ]:
%matplotlib inline
fib, ax = plt.subplots(subplot_kw=dict(projection='3d'))
ax.plot(latents[:, 0], latents[:, 1], latents[:, 2])

In [ ]:
%matplotlib inline
plt.plot(pca.components_.T);
plt.legend([f'PC{i} loading' for i in range(1,5)], loc='upper left')

In [ ]:
import pathlib
import datetime
from matplotlib.animation import FFMpegWriter, PillowWriter
import functools
import warnings

from common_cl_code.datasets import ArrayWithTime


class AnimationManager:
    """
    Examples
    --------
    >>> tmp_path = getfixture('tmp_path')  # this is mostly for the doctesting framework
    >>> with AnimationManager(outdir=tmp_path) as am:
    ...     for i in range(2):
    ...         for ax in am.axs.flatten():
    ...             ax.cla()
    ...         # animation things would go here
    ...         am.grab_frame()
    ...     fpath = am.outfile
    >>> assert fpath.is_file()
    """
    def __init__(self, outdir, filename_stem=None, n_rows=1, n_cols=1, fps=20, dpi=100, filetype="mp4", figsize=(10, 10), projection='rectilinear', make_axs=True, fig=None):
        outdir = pathlib.Path(outdir)


        if filename_stem is None:
            time_string = datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
            filename_stem = f"movie_{time_string}-{str(hash(id(self)))[-3:]}"

        self.filetype = filetype
        self.outfile = pathlib.Path(outdir).resolve() / f"{filename_stem}.{filetype}"
        Writer = FFMpegWriter
        if filetype == 'gif':
            Writer = PillowWriter
        if filetype == 'webm':
            Writer = functools.partial(FFMpegWriter, codec='libvpx-vp9')

        self.movie_writer = Writer(fps=fps, bitrate=-1)
        if fig is None:
            if make_axs:
                self.fig, self.axs = plt.subplots(n_rows, n_cols, figsize=figsize, layout='constrained', squeeze=False, subplot_kw={'projection': projection})
            else:
                self.fig = plt.figure(figsize=figsize, layout='constrained')
        else:
            self.fig = fig
        self.movie_writer.setup(self.fig, self.outfile, dpi=dpi)
        self.seen_frames = 0
        self.finished = False

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.seen_frames:
            self.finish()
        else:
            warnings.warn('closed without any frame grabs')

    def finish(self):
        if not self.finished:
            self.movie_writer.finish()
            self.finished = True

    def grab_frame(self):
        self.movie_writer.grab_frame()
        self.seen_frames += 1

    def display_video(self, embed=False, width=None):
        from IPython import display
        if self.filetype == 'gif':
            display.display(display.Image(self.outfile, embed=embed, width=width))
        else:
            display.display(display.Video(self.outfile, embed=embed, width=width))



def plot_history_with_tail(ax, data, current_t, tail_length=1, scatter_all=True, dim_1=0, dim_2=1, hist_bins=None, invisible=False, scatter_alpha=.1, scatter_s=5):
    """
    Examples
    --------
    >>> fig, ax = plt.subplots()
    >>> X = np.random.normal(size=(100,2))
    >>> X = ArrayWithTime.from_notime(X)
    >>> plot_history_with_tail(ax, data=X, current_t=75, tail_length=4, scatter_alpha=1)
    """
    ax.cla()

    s = np.ones_like(data.t).astype(bool)
    if scatter_all:
        s = data.t <= current_t
    if hist_bins is None:
        ax.scatter(data[s,dim_1], data[s,dim_2], s=scatter_s, c='gray', edgecolors='none', alpha= 0 if invisible else scatter_alpha)
        back_color = 'white'
        forward_color = 'C0'
    else:
        s = s & np.isfinite(data).all(axis=1)
        ax.hist2d(data[s,dim_1], data[s,dim_2], bins=hist_bins)
        back_color = 'black'
        forward_color = 'white'


    linewidth = 2
    size = 10
    s = (current_t - tail_length < data.t) & (data.t <= current_t)
    ax.plot(data[s, dim_1], data[s, dim_2], color=back_color, linewidth=linewidth * 1.5, alpha= 0 if invisible else 1)
    ax.scatter(data[s, dim_1][-1], data[s, dim_2][-1], s=size * 1.5, color=back_color, alpha= 0 if invisible else 1)
    ax.plot(data[s, dim_1], data[s, dim_2], color=forward_color, linewidth=linewidth, alpha= 0 if invisible else 1)
    ax.scatter(data[s,dim_1][-1], data[s,dim_2][-1], color=forward_color, s=size, zorder=3, alpha= 0 if invisible else 1)
    ax.axis('off')


In [ ]:
latents = ArrayWithTime(latents, a_bins[:-1])
stims = ArrayWithTime(st_c,st_t)


In [ ]:
unique_stims = np.sort(np.unique(stims))

In [ ]:
tail_length = 1
with AnimationManager(outdir='/home/jgould/Downloads/', filetype='mp4') as am:
    ax = am.axs[0,0]
    for current_t in np.arange(100, 120, .05):
        ax.cla()
        plot_history_with_tail(data=latents, current_t=current_t, tail_length=tail_length, ax=ax, scatter_alpha=.7)
        sub_stims = stims.slice_by_time(slice(current_t-tail_length, current_t))
        for stim_c, stim_t in zip(sub_stims, sub_stims.t):
            idx = latents.time_to_sample(stim_t)

            stim_c_id = np.argwhere(unique_stims == stim_c)[0][0]
            ax.scatter(latents[idx-1,0], latents[idx-1,1], c=f'C{stim_c_id}', s=30, label=f'{stim_c}')
        ax.set_title(f't={current_t:.2f}')
        ax.legend(loc='lower right')
        am.grab_frame()

In [ ]:
st_c